features = [lat, lon, zone_code, hemisphere_NS, hemisphere_WE]  

One Hot Encoding

StandardScaler()  

KMeans / DBSCAN / HDBSCAN  


In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import silhouette_score
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans
import plotly.express as px
import pandas as pd 
import numpy as np 
import os
import warnings

warnings.filterwarnings('ignore')
os.listdir('..\data')

['data_processed.csv', 'UPPLY-SEAPORTS.csv']

In [13]:
df = pd.read_csv("..\data\data_processed.csv", sep=',')
data_copia = df.copy()
df.sample(5)

,latitude,longitude,zone_code,hemisphere_NS,hemisphere_WE
10478,38.7333,16.1667,EU-SEU,North,East
9070,51.6333,-4.0833,EU-NEU,North,West
10677,52.1333,4.6667,EU-NEU,North,East
4690,50.9670,4.3830,EU-NEU,North,East
1519,65.4167,35.6833,EU-BLA,North,East


In [14]:
features_num = ["latitude", "longitude"]

features_cat = ["zone_code", "hemisphere_NS", "hemisphere_WE"]

In [15]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), features_num),
        ("cat", OneHotEncoder(), features_cat)
    ]
)

In [16]:
model = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("cluster", KMeans(n_clusters=41, random_state=42))
])

In [17]:
model.fit(df[features_num + features_cat])
df["cluster"] = model["cluster"].labels_

In [18]:
X = model["preprocess"].transform(df[features_num + features_cat])

silhouette_score(X, df["cluster"])

0.8253027733676683

In [19]:
model["cluster"].inertia_

836.2558261753996

In [20]:
df["cluster"].value_counts()

cluster
4     2857
0     1918
6     1431
5      780
13     688
8      665
10     638
17     504
9      475
11     447
12     310
14     247
24     244
2      231
20     229
16     219
18     194
1      194
23     182
3      153
21     141
26     136
22     136
25     136
34     122
36     114
27     112
29     109
32      96
19      76
15      72
40      56
35      47
39      46
31      44
28      44
7       40
30      39
38      38
37      34
33      24
Name: count, dtype: int64

In [21]:
df.groupby('cluster')[['latitude', 'longitude']].mean()

,latitude,longitude
cluster,,
0,50.761770,5.186898
1,-27.390806,-52.534350
2,23.748347,115.577438
3,16.854088,-66.546323
4,59.640817,17.958181
5,37.826865,-82.476621
6,53.915301,-4.919156
7,-29.443303,116.023360
8,40.901000,27.067540


In [22]:
fig = px.scatter_geo(
    df,
    lat="latitude",
    lon="longitude",
    color="cluster",
    hover_name="zone_code"
)

fig.show()